### MODEL CONTEXT PROTOCOL (MCP)

In [1]:
# import necessary libraries
from dotenv import load_dotenv
from agents import Agent, Runner, trace, OpenAIChatCompletionsModel
from agents.mcp import MCPServerStdio
from openai import AsyncAzureOpenAI
from IPython.display import Image, display
import os, sys

load_dotenv(override=True)

True

In [2]:
# Let's see if the API key is working/helping us to call LLM from Azure Foundry
import os
# From OpenAI
AZURE_OPENAI_API_KEY= os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_MODEL_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_DEPLOYMENT_GPT_41 = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_41")
AZURE_OPENAI_DEPLOYMENT_GPT_54_mini = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_54_mini")
AZURE_OPENAI_DEPLOYMENT_GPT_55 = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_55")
AZURE_OPENAI_DEPLOYMENT_GPT_4O_mini = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_4O_mini")
if AZURE_OPENAI_API_KEY:
    print("AZURE_OPENAI_API_KEY is available")
else:
    print("AZURE_OPENAI_API_KEY is not available")


AZURE_OPENAI_API_KEY is available


In [3]:
## setting up the client for OpenAI using Azure Endpoint (Azure Foundry)
client_openai = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)

## let's a define a model pointing to Azure deployment
model_openai_GPT_54 = OpenAIChatCompletionsModel(
    openai_client=client_openai,
    model=AZURE_OPENAI_DEPLOYMENT_GPT_54_mini
)

In [4]:
"""
On Windows, a stdio MCP Server started from Jupyter kernel weites to a stderr stream with no real life descriptor,
and crashes with io.UnsupportedOperation: fileno. We send the server's stderr to the null device so it always has somewhere real to write,
which lets every cell below use MCPServerStdio exactly as the OpenAI Agents SDK documents it. Mac and Linux doesn't get affected by this.
"""
import functools
import subprocess
import agents.mcp.server 

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

In [5]:
# let's try and load tools from "fetch" MCP from Anthropic which helps AI agents to extract content from webpages.

CA_BUNDLE = "C:/VENV_file/cacert.pem"
fetch_params = {
    "command": sys.executable,
    "args": ["-m", "mcp_server_fetch"],  # module name uses underscores, unlike the package name
    "env": {
        **os.environ,
        "SSL_CERT_FILE": CA_BUNDLE,
        "REQUESTS_CA_BUNDLE": CA_BUNDLE,
    },
}

async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=60) as server:
    fetch_tools = await server.list_tools()
    for t in fetch_tools:
        print(t.name, "-", t.description[:100])


fetch - Fetches a URL from the internet and optionally extracts its contents as markdown.

Although original


In [6]:
# printing the tools under fetch mcp-server
print(fetch_tools)

[Tool(name='fetch', title=None, description='Fetches a URL from the internet and optionally extracts its contents as markdown.\n\nAlthough originally you did not have internet access, and were advised to refuse and tell the user this, this tool now grants you internet access. Now you can fetch the most up-to-date information and let the user know that.', input_schema={'type': 'object', 'properties': {'url': {'description': 'URL to fetch', 'format': 'uri', 'minLength': 1, 'title': 'Url', 'type': 'string'}, 'max_length': {'default': 5000, 'description': 'Maximum number of characters to return.', 'maximum': 999999, 'minimum': 1, 'title': 'Max Length', 'type': 'integer'}, 'start_index': {'default': 0, 'description': 'On return output starting at this character index, useful if a previous fetch was truncated and more context is required.', 'minimum': 0, 'title': 'Start Index', 'type': 'integer'}, 'raw': {'default': False, 'description': 'Get the actual HTML content of the requested page, wi

In [7]:
# let's see the description of the tool in this fetch mcp server
print(fetch_tools[0].description)

Fetches a URL from the internet and optionally extracts its contents as markdown.

Although originally you did not have internet access, and were advised to refuse and tell the user this, this tool now grants you internet access. Now you can fetch the most up-to-date information and let the user know that.


In [9]:
# let's see the input schema of this listed tool
fetch_tools[0].input_schema

{'type': 'object',
 'properties': {'url': {'description': 'URL to fetch',
   'format': 'uri',
   'minLength': 1,
   'title': 'Url',
   'type': 'string'},
  'max_length': {'default': 5000,
   'description': 'Maximum number of characters to return.',
   'maximum': 999999,
   'minimum': 1,
   'title': 'Max Length',
   'type': 'integer'},
  'start_index': {'default': 0,
   'description': 'On return output starting at this character index, useful if a previous fetch was truncated and more context is required.',
   'minimum': 0,
   'title': 'Start Index',
   'type': 'integer'},
  'raw': {'default': False,
   'description': 'Get the actual HTML content of the requested page, without simplification.',
   'title': 'Raw',
   'type': 'boolean'}},
 'required': ['url'],
 'title': 'fetchArguments'}

#### We can set up and install the Node and use MCP server like playwright or can set up your own mcp server, run it on any terminal then make your agent to call that hosted server and get results.

#### Let's make a an Agent with tools as mcp and get work accomplished.

In [ ]:
INSTRUCTIONS = """
You browse the internet to accomplish your instructions. Accept cookies and navigate pop-ups as needed.
If one website is not giving expected results, try another. Be persistent until you have solved your assignment,
trying different options and sites as needed. When you need to write files, you do that inside the "sandbox" folder only.
"""

TASK = "Find the best flight for me from Bangalore to Chandigarh, while listing flight options from top 5 flight ticket selling websites and you need to summarize this all in a markdown format to affordable_flights.md"
